# CXAS SCRAPI
The CX Agent Studio Scripting API (CXAS SCRAPI) is a high-level API that extends the official Google [Python Client for CX Agent Studio](https://docs.cloud.google.com/customer-engagement-ai/conversational-agents/ps). CXAS SCRAPI makes using CX Agent Studio easier, more friendly, and more pythonic for bot builders, developers, and maintainers.

### What Can I Do With CXAS SCRAPI?
With CXAS SCRAPI you can perform many agent building and maintenance actions at scale including, but not limited to:
- Create, Update, Delete, Get, and List for all CXAS resources types (i.e. Apps, Agents, Tools, Guardrails, Deployments, Sessions, etc.)
- Run Evaluations and analyze Conversation History directly through code
- Convert robust Agent responses and configurations into unified, easy-to-read Python dictionaries and dataframes
- Orchestrate and test complex multi-agent setups locally
- Manage App-level Variables and configurations programmatically
- ...and much, much more!

---

* Authors: Patrick Marlow, Uma Nagarsekar, Suraj Dharmapuram, Sethu Sundararaman, Yatish Gupta, Alex Yu
* Last Major Update: `2026-03-27`

# Setup (once per runtime instance)

In [ ]:
# Authenticate
from google.colab import auth
auth.authenticate_user()

## Install cxas-scrapi

In [ ]:
!pip install cxas_scrapi --quiet

# Basic CRUD Ops
You can perform all basic CRUDL ops on the API from any of the core classes / resources in CXAS.

Below are some basic get / list ops.  
You can also create, update, delete as needed.

## App Client

In [1]:
from cxas_scrapi import Apps, Agents, Tools

In [ ]:
APP_NAME = "YOUR_FULLY_QUALIFIED_APP_NAME"
PROJECT_ID = APP_NAME.split("/")[1]
LOCATION = APP_NAME.split("/")[3]

# Inspect App
app_client = Apps(project_id=PROJECT_ID, location=LOCATION)
app = app_client.get_app(APP_NAME)

print(f"APP NAME: {app.name}")
print(f"APP DISPLAY NAME: {app.display_name}")
print(f"APP MODEL: {app.model_settings.model}")
print(f"# Variables in App: {len(app.variable_declarations)}")
print(f"Logging Enabled: {app.logging_settings.cloud_logging_settings.enable_cloud_logging}")

### Export App
You can export to `gcs_uri` or `local_path`<br>
You can also choose `export_format` as oneof `JSON`, `YAML` (defaults to JSON)

In [ ]:
res = app_client.export_app(APP_NAME, local_path="/content/sample_agent.zip")

### Import to Existing App
You can import an app from several locations:
- `app_content`, the raw bytes of the zip archive of the app
- `gcs_uri`, import from a GCS bucket/path
- `local_path`, import from a local file path

You can also define your conflict strategy:
- `REPLACE`, this will only replace items that are conflicting in the existing app
- `OVERWRITE`, this will replace the entire app (full overwrite)

In [ ]:
lro = app_client.import_app(
  APP_NAME,
  local_path="/content/sample_agent.zip",
  conflict_strategy="OVERWRITE"
  )
print(lro.result())

### Import as New App

In [ ]:
lro = app_client.import_as_new_app(
  display_name="SCRAPI NEW APP",
  local_path="/content/sample_agent.zip"
)
print(lro.result())

## Agent Client

In [ ]:
# Inspect Root Agent
agent_client = Agents(APP_NAME)
all_agents = agent_client.list_agents(APP_NAME)
root_agent = agent_client.get_agent(app.root_agent)

print(f"# Agents in App: {len(all_agents)}")
print(f"Root Agent: {root_agent.display_name}")
print(f"# Tools in {root_agent.display_name}: {len(root_agent.tools)}")

# Agents in App: 10
Root Agent: Shopping Assistant
# Tools in Shopping Assistant: 0


## Tools Client

In [ ]:
# Inspect Tools
tools_client = Tools(APP_NAME)
tools_map = tools_client.get_tools_map(APP_NAME, reverse=True)
print(f"Total Tools in App: {len(tools_map)}")

# Get Individual Tool
search_tool = tools_client.get_tool(tools_map["_search_products"])
print(search_tool.display_name)
print(str(search_tool.python_function)[:200])

Total Tools in App: 26
_search_products
name: "_search_products"
python_code: "from typing import List, Dict, Any\nimport re\n\ndef _search_products(query: str) -> List[Dict[str, Any]]:\n  \"\"\"Search products using whole-word matching to 


# Run / BidiRun Session Testing
You can directly test queries to your App by using the Sessions client.  
The `.run()` operation supports various input types, including text, events, multimodal inputs, etc.

Input args for `Session.run`:
- `session_id`: `str`
- `text`: `str | list[str] | None`
- `dtmf`: `str | None`
- `event`: `str | None`
- `event_vars`: `dict[str, Any] | None`
- `blob`: `bytes | None`
- `blob_mime_type`: `str = "application/octet-stream"`
- `variables`: `dict[str, Any] | None`
- `tool_responses`: `list[dict[str, Any]] | None`
- `audio`: `bytes | None`
- `audio_config`: `dict[str, Any] | None`
- `input_audio_config`: `dict[str, Any] | None`
- `output_audio_config`: `dict[str, Any] | None`
- `deployment_id`: `str | None`
- `historical_contexts`: `list[dict[str, Any]] | str | None`
- `turn_count`: `int | None`
- `modality`: `Modality | str = Modality.TEXT`
- `use_tool_fakes`: `bool = False`
- `background_noise_file`: `str | None`
- `burst_noise_files`: `list[str] | None`

For bidirectional streaming audio testing, the `.async_bidi_run_session()` method is available:
- `config`: `dict`
- `inputs`: `list[dict[str, Any]]`
- `background_noise_file`: `str | None`
- `bg_noise_snr`: `float = 15.0`

## Text Testing

In [ ]:
from cxas_scrapi import Sessions

session_client = Sessions(APP_NAME)

In [ ]:
inputs = ["hi!", "party planner"]
session_id = session_client.create_session_id()

for input in inputs:
    res = session_client.run(session_id, input)
    session_client.parse_result(res)

## Audio Testing
*Note: Because Colab is a headless environment, you cannot interactively speak to the agent using your microphone or hear its voice in real-time. Instead, the examples below demonstrate **programmatic audio testing**. SCRAPI automatically converts your text inputs into synthesized audio bytes using the Cloud Text-to-Speech API, streams them over WebSockets (simulating a live call), and parses the response to verify the agent's Speech-to-Text and voice reply paths. If you want to perform real-time, interactive bidirectional audio testing (using a physical microphone/speaker), you must run these methods (specifically `async_bidi_run_session`) in a local, non-Colab environment.*

In [ ]:
inputs = ["hi!", "party planner"]

session_id = session_client.create_session_id()
# Run audio session with single input
print("\n\nRunning audio session with single input")
for input in inputs:
    res = session_client.run(
        session_id,
        text=input,
        modality="audio"
    )
    session_client.parse_result(res)


session_id = session_client.create_session_id()
# Run audio session with list of inputs
print("\n\nRunning audio session with list of inputs")
res = session_client.run(
    session_id,
    text=inputs,
    modality="audio"
)
session_client.parse_result(res)



Running audio session with single input




Running audio session with list of inputs


# Offline Eval Ops
* **What it is:** Programmatic tests run locally via `cxas-scrapi` (e.g., Python scripts, Colab notebooks) that simulate targeted conversations.
* **When to use:** During active development. They provide immediate, deterministic feedback to validate that your agent correctly triggers specific tools, respects guardrails, and routes intents properly before you ever merge your code.

## Turn Evals
Ideal for rapid, stateless testing. You define a single user input in a YAML payload and assert exactly what the agent should do in response (e.g., matching text, triggering a specific tool, or transferring to a sub-agent).

In [ ]:
from cxas_scrapi import TurnEvals

turn_evals = TurnEvals(APP_NAME)

# Example YAML definition strictly testing a stateless single-turn interaction
yaml_payload = """
tests:
  - name: "Welcome Event Check"
    user: "<event>WelcomeEvent</event>"
    expectations:
      - type: "contains"
        value: "Shopping Assistant"

  - name: "Party Planner transfer"
    user: "Plan a party for my 4 year old"
    expectations:
      - type: "agent_transfer"
        value: party_planner

  - name: "Product search tool I/O"
    user: "cupcake decorations"
    variables:
      first_turn: true
    expectations:
      - type: "tool_called"
        value: "search_products"
      - type: "tool_input"
        value:
          query: "cupcake decorations"
      - type: "tool_output"
        value:
          search_products:
            result: {}
"""

# Loads the TurnTestCase Pydantic array
test_cases = turn_evals.load_turn_test_cases_from_yaml(yaml_payload)

# Executes exactly one session turn per test case, printing failures and returning a dataframe of results
results_df = turn_evals.run_turn_tests(test_cases, debug=False)
display(results_df)

Running Turn Test: Welcome Event Check
SUCCESS: Welcome Event Check
------------------------------
Running Turn Test: Party Planner transfer
SUCCESS: Party Planner transfer
------------------------------
Running Turn Test: Product search tool I/O
SUCCESS: Product search tool I/O
------------------------------


,test_name,user,status,errors
0,Welcome Event Check,<event>WelcomeEvent</event>,SUCCESS,
1,Party Planner transfer,Plan a party for my 4 year old,SUCCESS,
2,Product search tool I/O,cupcake decorations,SUCCESS,


## Simulation Evals
Built for stateful, multi-turn scenario testing. Use the `UserSimulator` and an LLM driven "user persona" to automatically converse with your agent, navigating complex flows across multiple turns to validate conversational memory and context retention.

In [ ]:
from cxas_scrapi import SimulationEvals

# Define the simulated test scenario (steps with goals and criteria)
test_case = {
    "steps": [
        {
            "goal": "Greet the agent",
            "success_criteria": "The agent acknowledges the greeting",
            "max_turns": 2
        },
        {
            "goal": "Tell the agent that you want to plan a party for your 4 year old that loves unicorns and rainbows",
            "success_criteria": "The agent generates some party ideas for you to choose from.",
            "max_turns": 3
        },
    ]
}

# Initialize the Simulator
sim_evals = SimulationEvals(APP_NAME)
results = sim_evals.simulate_conversation(test_case=test_case, console_logging=False)
results.generate_report()

## Tool Evals
One feature exclusive to CXAS SCRAPI is the ability to author unit tests for tools so you can programmatically test them to ensure they are working properly.

Focuses strictly on your backend integrations. Use this to directly invoke your tools, validating that APIs resolve correctly, payloads are formatted properly, and schemas are strictly adhered to, completely isolated from the agent's LLM.

This utilizes the same methods above for `execute_tool()` but also allows you to define expectations to evaluate your tool outputs.

* Supported Tool types
  1. Open API Toolset
  2. Python tools
  3. Datastore tools
  4. Google search tool

* Supported context
  1. Variables

* Supported Validations
  1. EQUALS
  2. CONTAINS
  3. GREATER_THAN
  4. LESS_THAN
  5. LENGTH_EQUALS
  6. LENGTH_GREATER_THAN
  7. LENGTH_LESS_THAN
  8. IS_NULL
  9. IS_NOT_NULL

### Auto Generate Tool Evals
To bootstrap tool unit tests, we provide a simple auto generate method that will produce full functional unit tests directly from your Agent application. You can choose to mine conversation data to populate the test data, or you can simply generate templates and populate them yourself.

Generates configurable YAML test templates for tools defined in the app. Parses the application's tools and Conversation History to try and intelligently scaffold the tool unit tests.

Args:
- `target_dir`: The directory path where the generated YAML files - will be saved. Defaults to 'tool_tests'.
- `include_tools`: An optional list of tool display names to restrict the generation. If None, all tools in the app are evaluated.
- `exclude_tools`: An optional list of tool display names (or prefixes) to exclude from generation. Matches if a tool's display name starts with any string in this list.
- `overwrite`: If True, existing YAML test templates in the target directory will be overwritten. If False, existing files are skipped.
- `mine_tool_data`: If True, queries recent conversations to populate generated tests with real tool payload arguments.
- `mine_conversations_limit`: The maximum number of conversations to scan when mining real tool arguments.

In [ ]:
from cxas_scrapi import ToolEvals

tool_evals = ToolEvals(APP_NAME)

TEST_DIR = "/content/tool_tests"

tool_evals.generate_tool_tests(
  target_dir=TEST_DIR,
  mine_tool_data=True,
  overwrite=True,
  exclude_tools=["_list_products"]
  )

Generated test template: /content/tool_tests/generate_order_summary.yaml
Generated test template: /content/tool_tests/get_seasonal_party_bundles.yaml
Generated test template: /content/tool_tests/optimize_budget.yaml
Generated test template: /content/tool_tests/handle_support.yaml
Generated test template: /content/tool_tests/optimize_party_budget.yaml
Generated test template: /content/tool_tests/list_products.yaml
Generated test template: /content/tool_tests/get_themes.yaml
Generated test template: /content/tool_tests/analyze_damage_image.yaml
Generated test template: /content/tool_tests/_get_products_for_theme.yaml
Generated test template: /content/tool_tests/_get_themes.yaml
Generated test template: /content/tool_tests/_search_products.yaml
Generated test template: /content/tool_tests/filter_themes.yaml
Generated test template: /content/tool_tests/check_party_order_status.yaml
Generated test template: /content/tool_tests/generate_theme_image.yaml
Generated test template: /content/tool

### Run Tool Evals

In [ ]:
test_cases = tool_evals.load_tool_tests_from_dir(TEST_DIR)
results = tool_evals.run_tool_tests(test_cases)

num_failures = (results['status'] == 'FAILURE').sum()
num_success = (results['status'] == 'SUCCESS').sum()

print(f"##### Total Tests: {len(results)} #####")
print(f"\nTests Passed: {num_success}")
print(f"Tests Failed: {num_failures}")
print(f"Percentage Passing: {num_success/len(results):.2f}")

Running test: generate_theme_image_test_1 (generate_theme_image)
FAILED: generate_theme_image --> generate_theme_image_test_1
["Response expectation failed: path='$.prompt', actual='None', expected='None', operator='Operator.IS_NOT_NULL'", "Response expectation failed: path='$.success', actual='None', expected='None', operator='Operator.IS_NOT_NULL'", "Response expectation failed: path='$.error', actual='None', expected='None', operator='Operator.IS_NOT_NULL'"]
------------------------------
Running test: optimize_party_budget_test_1 (optimize_party_budget)
FAILED: optimize_party_budget --> optimize_party_budget_test_1
["Response expectation failed: path='$.result', actual='None', expected='PASSED', operator='Operator.CONTAINS'"]
------------------------------
Running test: optimize_budget_test_1 (optimize_budget)
FAILED: optimize_budget --> optimize_budget_test_1
["Response expectation failed: path='$.result', actual='None', expected='PASSED', operator='Operator.CONTAINS'"]
----------

In [ ]:
# Inspect the results in a dataframe
results.head(10)

,test_name,tool,status,errors
0,generate_theme_image_test_1,generate_theme_image,FAILED,"Response expectation failed: path='$.prompt', ..."
1,optimize_party_budget_test_1,optimize_party_budget,FAILED,"Response expectation failed: path='$.result', ..."
2,optimize_budget_test_1,optimize_budget,FAILED,"Response expectation failed: path='$.result', ..."
3,get_themes_test_1,get_themes,PASSED,
4,generate_order_summary_test_1,generate_order_summary,FAILED,"Response expectation failed: path='$.result', ..."
5,list_products_test_1,list_products,FAILED,"Response expectation failed: path='$.result', ..."
6,process_instant_exchange_test_1,process_instant_exchange,FAILED,"Response expectation failed: path='$.result', ..."
7,check_party_order_status_test_1,check_party_order_status,FAILED,"Response expectation failed: path='$.result', ..."
8,process_party_payment_test_1,process_party_payment,FAILED,"Response expectation failed: path='$.result', ..."
9,vertex_search_datastores_search_test_1,vertex_search_datastores_search,FAILED,"Response expectation failed: path='$.result', ..."


### Manually Write Tool Evals
For easy authoring format, we provide a YAML template to quickly define tests that you want to run. This format will work for all tool types currently supported.<br>

We provide several methods to ingest your test cases:
- `load_tool_test_cases_from_yaml`, expects a YAML formatted string in the format provided below
- `load_tool_test_cases_from_data`, expects a JSON formatted string using the same structure as the YAML format
- `load_tool_test_cases_from_file`, expects a filepath as a string and parses the incoming file into the test case format
- `load_tool_tests_from_dir`, expects a dirpath as a string and loads all files from the director in test case format

In [ ]:
yaml_test_string = """
tests:
  # Option 1: Using existing app variables
  - name: test_existing_variable
    tool: _search_products
    args:
      query: "cupcakes"
    variables:
      - products
      - catalog
    expectations:
      response:
        - path: "$.result"
          operator: "is_not_null"
        - path: "$.result"
          operator: "length_greater_than"
          value: 0

  # Option 2: Overriding with custom mock data
  - name: test_custom_variable
    tool: _search_products
    args:
      query: "balloons"
    variables:
      products:
        - title: "Mock Balloon"
          subtitle: "A fake balloon for testing"
          price: 10
          in_stock: true
    expectations:
      response:
        - path: "$.result[0].title"
          operator: "equals"
          value: "Mock Balloon"
      variables:
        - path: "$.products[0].title"
          operator: "equals"
          value: "Mock Balloon"
"""

In [ ]:
# Load the test case and run it
test_cases = tool_utils.load_tool_test_cases_from_yaml(yaml_test_string)
results = tool_utils.run_tool_tests(test_cases, debug=False)

Running test: test_existing_variable (_search_products)
SUCCESS: _search_products --> test_existing_variable
------------------------------
Running test: test_custom_variable (_search_products)
SUCCESS: _search_products --> test_custom_variable
------------------------------


### Execute Tool w/ Variables
Similar to the "Test Tool" functionality in the UI, you can execute various types of tools via the API and provide input Variables to the tool.

When providing Variables to the tool execution you have a few options:
- `variables=None`, this is the default setting and will pass in all declared Variables that are currently set on the App
- `variables=["var_name1", "var_name2"]`, this allows you to filter to only the variable names that you provide. Internally, we will look up the variable value for you and send it to the tool
- `variables={"var_name": "my_custom_value", "other_var": 1234}`, this allows you to define any arbitrary set of variables to pass to the tool. Typically this would be used to override the default variable values on the app

In [ ]:
from cxas_scrapi import Tools

tools_client = Tools(APP_NAME)

tools_map = tools_client.get_tools_map(APP_NAME, reverse=True)
print(f"# of Tools in App: {len(tools_map)}\n")
list(tools_map.items())[:5] # show the first few tools in the map

In [ ]:
from cxas_scrapi import Tools

tool_client = Tools(APP_NAME)

args = {"query": "cupcakes"}

res = tool_client.execute_tool(APP_NAME, "_search_products", args=args, variables=None)
str(res)[:200]

## Callback Evals
Validates your deterministic callback logic. Test your [Callback](https://docs.cloud.google.com/customer-engagement-ai/conversational-agents/ps/callback) handlers to ensure your application correctly manipulates the payload, injects context, or overrides responses before the LLM ever sees the prompt.

In [ ]:
# callback eval placeholder
from cxas_scrapi import Apps, CallbackEvals

GCS_BUCKET = "gs://YOUR_GCS_BUCKET"
GCS_FILE_PATH = "YOUR/FILE/PATH.zip"
GCS_URI = f"{GCS_BUCKET}/{GCS_FILE_PATH}"

app_client = Apps(PROJECT_ID, LOCATION)

# Export App and Download to Local Colab
app_client.export_app(app_id=APP_NAME, gcs_uri=GCS_URI)
!gcloud storage cp GCS_URI /content/local_agent.zip
!unzip /content/local_agent.zip

In [ ]:
# Authoring Callback Tests and Write to Local File
%%writefile /content/LOCAL_AGENT/agents/My_Cool_Agent/before_model_callbacks/before_model_callbacks_01/test.py

from cxas_scrapi.utils.callback_libs import CallbackContext, LlmRequest, Content, Part, LlmResponse
from python_code import before_model_callback

def test_welcome_event_triggers_greeting():
    # 1. Arrange: Setup the "fake" state matching your condition
    mock_context = CallbackContext(
        state={"first_turn": True, "no_input_counter": 0},
        user_content=Content(parts=[Part(text="<event>WelcomeEvent</event>")])
    )
    mock_request = LlmRequest()

    # 2. Act: Run the callback
    response = before_model_callback(mock_context, mock_request)

    # 3. Assert: Verify the override happened correctly
    assert response is not None
    assert response.content.parts[0].text == "👋 Hey I'm your Shopping Assistant! What can I help you with today?"
    print("✅ Welcome Event Test Passed!")

def test_ignores_regular_utterances():
    # 1. Arrange: Setup state where input is just "Hello", not the WelcomeEvent
    mock_context = CallbackContext(
        state={"first_turn": True, "no_input_counter": 0},
        user_content=Content(parts=[Part(text="Hello! I need to buy a gift.")])
    )
    mock_request = LlmRequest()

    # 2. Act: Run the callback
    response = before_model_callback(mock_context, mock_request)

    # 3. Assert: Verify no override happened (should return None)
    assert response is None
    print("✅ Ignore Regular Utterance Test Passed!")


In [ ]:
callback_evals = CallbackEvals()
callback_evals.run_callback_tests("/content/LOCAL_AGENT")

,agent_name,callback_type,test_name,status,error_message
0,Shopping_Assistant,before_model_callbacks,test_welcome_event_triggers_greeting,PASSED,None
1,Shopping_Assistant,before_model_callbacks,test_ignores_regular_utterances,PASSED,None


## Guardrails Evals
Designed for security tuning. Run sample datasets of safe and malicious prompts against your offline Guardrail configurations to calibrate refusal thresholds, block prompt injection, and fine-tune your safety policies without exposing live traffic.

Implements a new type of testing to explicitly check Guardrail outputs.  
Utilizes the new Google Sheets class to read/write data from Google Sheets.

Test Sheet: [CXAS SCRAPI - Guardrail Sample Dataset](https://docs.google.com/spreadsheets/d/1HxpuIB_vHQ2-LuJBLNpqMUCAtHR8biYjycj1AqR6AXA/edit?gid=0#gid=0)

In [ ]:
from cxas_scrapi import GoogleSheetsUtils, GuardrailEvals

sheets_client = GoogleSheetsUtils()
SHEET_NAME = "CXAS SCRAPI - Guardrail Sample Dataset"

df = sheets_client.sheets_to_dataframe(SHEET_NAME, worksheet_name="input_dataset")

In [ ]:
guard_utils = GuardrailEvals(app_name=APP_NAME)
results_df = guard_utils.run_guardrail_tests(df, console_logging=True)
sheets_client.dataframe_to_sheets(results_df, sheet_name=SHEET_NAME, worksheet_name="results")

# Generate Summary and Append to Sheet
summary_df = guard_utils.generate_report(results_df)
sheets_client.append_dataframe_to_sheets(summary_df, sheet_name=SHEET_NAME, worksheet_name="summary")

Running Guardrail Tests: 100%|██████████| 12/12 [00:15<00:00,  1.32s/it]



######## Test Results ########

FAILURE: safety-test-1
["Expected trigger: False, Actual trigger: True"]
SUCCESS: safety-test-2
FAILURE: safety-test-3
["Expected trigger: False, Actual trigger: True"]
FAILURE: safety-test-4
["Expected trigger: False, Actual trigger: True"]
FAILURE: safety-test-5
["Expected trigger: False, Actual trigger: True"]
FAILURE: safety-test-6
["Expected trigger: False, Actual trigger: True"]
FAILURE: safety-test-7
["Expected trigger: False, Actual trigger: True"]
SUCCESS: safety-test-8
FAILURE: safety-test-9
["Expected trigger: False, Actual trigger: True"]
SUCCESS: safety-test-10
SUCCESS: safety-test-11
FAILURE: safety-test-12
["Expected guardrail type matching 'Blocklist', but got 'LlmPolicy'"]

######## Summary ########
Total Tests: 12 | Passed: 4 | Failed: 8



# Online Eval Ops
* **What it is:** The platform's native, asynchronous evaluation pipelines that automatically analyze conversations and generate quality metrics in the UI.
* **When to use:** After deployment. They are designed for continuous, high-level monitoring of conversations to track overarching quality metrics like hallucination rates, safety violations, and overall conversation success.

## Inspect Eval Metric Thresholds
There are various metrics associated with evals, and we can quickly lookup the current threshold settings before running our evals.

In [ ]:
from cxas_scrapi import Evaluations

eval_client = Evaluations(APP_NAME)
eval_settings = eval_client.get_evaluation_thresholds(print_console=True)

===== GLOBAL Settings =====
Hallucinations: DISABLED

===== GOLDEN Settings =====
Hallucinations: DISABLED

### Turn Level Metrics ###
- semantic_similarity_success_threshold: 1/4
- overall_tool_invocation_correctness_threshold: 0.8/1.0

===== SCENARIO Settings =====
Hallucinations: DISABLED




## Get Evals Map
Throughout CXAS SCRAPI there are various `get_x_map()` methods that help the developer quickly manage their resources by providing an O(1) lookup for Display Names and IDs.<br>
This is especially helpful for filtering resources to only ones that you care about.


In [ ]:
from cxas_scrapi import Evaluations

eval_client = Evaluations(APP_NAME)
evals_map = eval_client.get_evaluations_map(reverse=True)
list(evals_map["goldens"].items())[:5] # show the first 5 golden evals in the map

## Search and Filter Evals
We provide advanced filtering of your Evals so it's much easier to know which specific Evals are affected by specific resource changes.

You can filter on the following resources:
- Tools
- Variables
- Agents

The default search and filter behavior is to find Evals where the resource name is contained somewhere in the eval (e.g. in some turn or expectation).

**NOTE**: On the first search after class instantiation, a `search_index` will be built which can take 5-10 seconds.<br> Subsequent searches should be insant due to the cached index.

In [ ]:
# Search by Tools
res = eval_client.search_evaluations(APP_NAME, tools=["filter_themes"])
print(f"# of Evals with this Tool: {len(res)}")
print(res[:5]) # show subset of list

# of Evals with this Tool: 2
['HERO DEMO', 'Dino Dig - solo theme query']


In [ ]:
# Search by Variables
res = eval_client.search_evaluations(APP_NAME, variables=["products"])
print(f"# of Evals with this Variable: {len(res)}")
print(res[:5]) # show subset of list

# of Evals with this Variable: 2
['Basic Product Search', 'HERO DEMO']


In [ ]:
# Search by Agent
res = eval_client.search_evaluations(APP_NAME, agents=["Budget Optimizer"])
print(f"# of Evals with this Agent: {len(res)}")
print(res[:5]) # show subset of list

# of Evals with this Agent: 1
['HERO DEMO']


## Inspect an individual Eval

In [ ]:
hero_eval = eval_client.get_evaluation(evals_map["goldens"]["HERO DEMO"]) # Scenario based eval

print(f"# EVAL Turns: {len(hero_eval.golden.turns)}")
print(f"Last Eval Run Result: {hero_eval.last_completed_result.evaluation_status.name}\n")
print(str(hero_eval)[:500]) # show subset of eval proto

## Export Evals
Currently we support exporting Evals to the following formats:
- YAML (default)
- JSON

Files can be stored locally and modified before importing back to the App.

In [ ]:
hero_demo_eval_yaml = eval_client.export_evaluation(evals_map["goldens"]["HERO DEMO"], output_format="yaml")

print(hero_demo_eval_yaml[:200])

name: HERO DEMO
turns:
- user_event: WelcomeEvent
- agent: 👋 Hey I'm your Shopping Assistant! What can I help you with today?
- user: Birthday Party planner
- agent: '**Let''s make this birthday unfor


## Export Evals to Local Files

In [ ]:
eval_client.bulk_export_evals(APP_NAME, eval_type="goldens", output_dir="/content/")

Fetching evaluations map...
Found 3 goldens. Starting export to /content/evals...
✅ Exported: Basic_Product_Search.yaml
✅ Exported: HERO_DEMO.yaml
✅ Exported: Dino_Dig_-_solo_theme_query.yaml

Done! Successfully exported 3/3 goldens.


## Golden Eval YAML Format (Compressed)

In [ ]:
yaml_string = """
conversations:
- conversation: Cupcake Search
  tags: [product_search, p0]
  turns:
  - user:  <event>WelcomeEvent</event>
    agent: 👋 Hey I'm your Shopping Assistant! What can I help you with today?
  - user: cupcakes
    agent:  I've pulled up 6 items that match what you're looking for.
    tool_calls:
    -
      {
        "action": "search_products",
        "input_action_parameters": {
            "query": "cupcakes"
        }
      }
  expectations:
  - There must be a transfer_to_agent tool call that goes to the party_planner agent
"""

CUPCAKE_EVAL = "/content/evals/cupcake_eval.yaml"

with open(CUPCAKE_EVAL, "w") as f:
    f.write(yaml_string)

## Load Golden Eval from YAML

In [ ]:
from cxas_scrapi import EvalUtils

eval_utils_client = EvalUtils(APP_NAME)

In [ ]:
eval_dict = eval_utils_client.load_golden_eval_from_yaml(CUPCAKE_EVAL)
eval_dict

## Create and Run Golden Evaluation from YAML

In [ ]:
result = eval_utils_client.create_and_run_evaluation_from_yaml(CUPCAKE_EVAL)
result

In [ ]:
# Check if done
result["run"].done()

True

## Selectively Run Evals on Target Agent
CXAS SCRAPI provides many convenience methods for running and analyzing eval results.  
Here, you can selectively run evals based on their display name or types.

In [ ]:
from cxas_scrapi import EvalUtils

eval_utils_client = EvalUtils(APP_NAME)

# Choose Evals to Run
evals_to_run = ["Basic Product Search"]

In [ ]:
# Run Evals
eval_op = eval_utils_client.run_evaluation(evaluations=evals_to_run, app_id=APP_NAME)

# Wait for completion and get response
if hasattr(eval_op, 'result'):
    eval_response = eval_op.result()
else:
    eval_response = eval_op

print("Evaluation run completed successfully.")

run_results = eval_client.list_evaluation_results_by_run(eval_response.evaluation_run)

Evaluation run completed successfully.


## Eval Reporting
Parsing through all of the details for Evals in the UI can be daunting, so we provide some hierarchical convenience methods to summarize the data you need most at the detail you require.

There are 3 levels of detail:
- [1] Overall Summary
- [2] Detailed Failures
- [3] Trace View

Each view is progressively more detailed than the previous.

The primary method is `evals_to_dataframe` which will produce 3 distinct dataframes for viewing:
- `summary`, gives you the overall summary view
- `failures`, gives you the turn by turn failures to focus on, filtering out any other detail
- `trace`, gives you the full conversation view turn by turn, including pass and fail turns

In [ ]:
res = eval_client.list_evaluation_results(evals_to_run[0])
dfs = eval_utils_client.evals_to_dataframe(res)

In [ ]:
dfs["summary"]

,display_name,eval_result_id,evaluation_status,semantic_score,tool_invocation_score,create_time,update_time
0,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,❌ FAILED,None,None,2026-03-05 21:12:45.269336+00:00,NaT
1,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,❌ FAILED,None,None,2026-03-05 19:29:40.808261+00:00,NaT
2,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,❌ FAILED,None,None,2026-03-02 17:25:42.068367+00:00,NaT
3,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,❌ FAILED,None,None,2026-02-27 22:32:46.192412+00:00,NaT
4,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,❌ FAILED,None,None,2026-02-27 22:23:53.765773+00:00,NaT


In [ ]:
dfs["failures"].head()

,display_name,eval_result_id,turn_index,failure_type,expected,actual,score
0,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,2,Tool Call,product_carousel,product_carousel,0.0
1,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,4,Tool Call,product_carousel,product_carousel,0.0
2,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,4,Tool Call,product_carousel,product_carousel,0.0
3,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,2,Tool Call,product_carousel,product_carousel,0.0
4,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,3,Tool Call,product_carousel,product_carousel,0.0


In [ ]:
dfs["trace"].head()

,display_name,eval_result_id,turn_index,latency_seconds,semantic_score,semantic_outcome,hallucination_score,hallucination,tool_invocation_score,expectation_outcomes
0,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,1,None,4,PASS,-1,Unknown,0.0,"[{""expectation"": {""agent_response"": {""role"": ""..."
1,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,2,None,4,PASS,1,Justified,0.0,"[{""expectation"": {""agent_transfer"": {""target_a..."
2,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,3,None,4,PASS,1,Justified,0.0,"[{""expectation"": {""tool_call"": {""id"": ""adk-84d..."
3,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,4,None,4,PASS,1,Justified,0.0,"[{""expectation"": {""tool_call"": {""id"": ""adk-598..."
4,Basic Product Search,projects/google.com:nextgen-sandbox2/locations...,1,None,4,PASS,-1,Unknown,0.0,"[{""expectation"": {""agent_response"": {""role"": ""..."


# Latency Metrics
CXAS SCRAPI provides a simple way to gather latency metrics for either Evaluations or Conversation History.

Each output dataframe provides details like `count`, `average`, `p50`, `p90`, and `p99` latency metrics.

## Eval Latency Metrics
Optional Inputs:
- [1] You can provide a list of Eval display names
- [2] You can provide a list of previous eval run results

The details of the runs will be parsed against the conversation history to extract all details about latency for Tools, Callbacks, Guardrails, and the Eval run itself.

8 unique sets of data are then output:
- `eval_summary` and `eval_details`, Aggregated and detailed run stats
- `tool_summary` and `tool_details`, Stats grouped by `tool_name`
- `callback_summary` and `callback_details`, Grouped by `agent`, `stage` and `description`
- `guardrail_summary` and `guardrail_details`, Grouped by `agent` and `name`

In [ ]:
# Gather all of the latency metrics for Evals
df = eval_utils_client.get_latency_metrics_dfs(eval_names=evals_to_run)

Fetching 5 conversation traces for detailed latency metrics...


In [ ]:
df["eval_summary"].head(2)

,display_name,eval_result_id,evaluation_type,Average (Turn),p50 | p90 | p99 (Turn),Average (LLM),p50 | p90 | p99 (LLM),Average (Tool Call),p50 | p90 | p99 (Tool Call),Average (Guardrail),p50 | p90 | p99 (Guardrail),Average (Callback),p50 | p90 | p99 (Callback)
0,Basic Product Search,087dd036-1327-401c-8492-c8bf85fc7fd4,Golden,731 ms,803 ms | 1075 ms | 1174 ms,599 ms,478 ms | 802 ms | 875 ms,81 ms,77 ms | 97 ms | 102 ms,111 ms,0 ms | 268 ms | 328 ms,159 ms,182 ms | 211 ms | 222 ms
1,Basic Product Search,a08a63cf-4d8d-49fc-aaff-eacbc2af583e,Golden,780 ms,811 ms | 1227 ms | 1379 ms,623 ms,492 ms | 839 ms | 917 ms,95 ms,88 ms | 106 ms | 110 ms,155 ms,0 ms | 372 ms | 456 ms,176 ms,200 ms | 250 ms | 268 ms


In [ ]:
df["eval_details"].head()

,display_name,eval_result_id,turn_index,Total Turn Latency (ms),Tool Call Latencies (ms),LLM Latencies (ms),Guardrail Latencies (ms),Callback Latencies (ms),tool_names
0,Basic Product Search,087dd036-1327-401c-8492-c8bf85fc7fd4,1,134,0,0,0,47,
1,Basic Product Search,087dd036-1327-401c-8492-c8bf85fc7fd4,2,1186,65,883,335,181,"search_products, product_carousel"
2,Basic Product Search,087dd036-1327-401c-8492-c8bf85fc7fd4,3,791,77,478,0,183,"search_products, product_carousel"
3,Basic Product Search,087dd036-1327-401c-8492-c8bf85fc7fd4,4,816,102,435,0,223,"search_products, product_carousel"
4,Basic Product Search,a08a63cf-4d8d-49fc-aaff-eacbc2af583e,1,104,0,0,0,34,


***NOTE - The Eval metrics also include the following 6 dataframe that will be shown below, but we will show them only once for brevity***

## Conversation History Latency Metrics
With latency metrics from convo history, you can provide several filters:
- `time_filter`, you can provide values like `1d`, `2w`, `1m` etc. to filter to specific dates
- `source_filter`, accepts oneof `LIVE`, `SIMULATOR`, `EVAL`
- `limit`, the number of conversations to grab for the given timeframe (Default: 50)

The details of the runs will be parsed against the conversation history to extract all details about latency for Tools, Callbacks, Guardrails, and the Eval run itself.

6 unique sets of data are then output:
- `tool_summary` and `tool_details`, Stats grouped by `tool_name`
- `callback_summary` and `callback_details`, Grouped by `agent`, `stage` and `description`
- `guardrail_summary` and `guardrail_details`, Grouped by `agent` and `name`

In [ ]:
from cxas_scrapi import ConversationHistory

ch_client = ConversationHistory(APP_NAME)

df = ch_client.get_latency_metrics_dfs(APP_NAME, time_filter="7d", source_filter="LIVE", limit=300)

Fetching 129 conversation traces for detailed latency metrics...


In [ ]:
df["tool_summary"].head()

,tool_name,count,Average (ms),p50 (ms),p90 (ms),p99 (ms)
0,transfer_to_agent,164,0,0,1,1
1,get_themes,57,97,82,100,450
2,_get_themes,57,59,50,67,259
3,filter_themes,55,30,25,38,116
4,_get_products_for_theme,49,75,58,101,269


In [ ]:
df["callback_summary"].head()

,agent,stage,description,count,Average (ms),p50 (ms),p90 (ms),p99 (ms)
0,party_planner,BeforeModel,Handles the following:\n1. Init Agent greeting...,503,60,47,104,261
1,Shopping Assistant,BeforeModel,,268,60,44,77,398
2,party_planner,AfterTool,Debug for product carousel,226,37,30,51,150
3,party_planner,BeforeTool,Handles adding base_image bytes to tool call i...,226,34,29,44,143
4,Checkout,BeforeModel,Handles building the order_summary component d...,96,61,54,92,120


In [ ]:
df["guardrail_summary"].head()

,agent,name,count,Average (ms),p50 (ms),p90 (ms),p99 (ms)
0,party_planner,Prompt Guardrail 1767889897322,339,399,415,618,905
1,Shopping Assistant,Prompt Guardrail 1767889897322,167,401,413,597,847
2,Budget Optimizer,Prompt Guardrail 1767889897322,55,356,408,565,803
3,Checkout,Prompt Guardrail 1767889897322,49,384,425,612,836
4,Support,Prompt Guardrail 1767889897322,22,373,401,463,567


# Deployments Management
Manage your app deployments (environments and versions).

In [ ]:
from cxas_scrapi import Deployments

dep_client = Deployments(APP_NAME)

# List deployments
deployments = dep_client.list_deployments()
print(f"Deployments: {len(deployments)}")

# Guardrails Management
Manage Guardrail resources on the server (CRUD ops).

In [ ]:
from cxas_scrapi import Guardrails

gr_client = Guardrails(APP_NAME)

# List guardrails
guardrails = gr_client.list_guardrails()
print(f"Guardrails: {len(guardrails)}")

# App Variables Management
Manage global variables within the app.

In [ ]:
from cxas_scrapi import Variables

var_client = Variables(APP_NAME)

# List app variables
variables = var_client.list_variables()
print(f"App Variables: {len(variables)}")

# Changelogs
Audit logs of changes made to the app.

In [ ]:
from cxas_scrapi import Changelogs

ch_client = Changelogs(APP_NAME)

# List changelogs
logs = ch_client.list_changelogs()
print(f"Changelogs: {len(logs)}")

# Scorecards Management
Manage CCAI Insights QA Scorecards (Project-level resource).

In [ ]:
from cxas_scrapi.core.scorecards import Scorecards

# Scorecards take PROJECT_ID instead of APP_NAME
sc_client = Scorecards(PROJECT_ID)

# List scorecards
scorecards = sc_client.list_scorecards()
print(f"Scorecards: {len(scorecards)}")